In [ ]:
import os
from dotenv import load_dotenv
import weaviate
from weaviate.classes.init import Auth
from weaviate.classes.config import Property, DataType
from weaviate.classes.query import Filter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_weaviate.vectorstores import WeaviateVectorStore
from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore
from langchain_core.retrievers import BaseRetriever
from IPython.display import Markdown, display
from pydantic import BaseModel, Field
from typing import List, Literal

from qdrant_client.models import (
    Distance,
    VectorParams
)

from langchain_groq import ChatGroq

from langchain.chains import (
    create_history_aware_retriever,
    create_retrieval_chain
)

from langchain.chains.combine_documents import (
    create_stuff_documents_chain
)

from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)


from langchain_community.chat_message_histories import (
    RedisChatMessageHistory
)

from langchain_core.runnables.history import (
    RunnableWithMessageHistory
)

In [ ]:
# API Keys Setup

load_dotenv()

weaviate_url = os.getenv("weaviate_url")
weaviate_api_key = os.getenv("weaviate_api_key")
qdrant_url= os.getenv("qdrant_url")
qdrant_api_key= os.getenv("qdrant_api_key")
groq_api_key= os.getenv("groq_api_key")
redis_url = os.getenv("redis_url")

In [ ]:
# Connecting to Vector DB for Sementic Memory

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=weaviate_url,
    auth_credentials=weaviate.auth.AuthApiKey(weaviate_api_key),
    skip_init_checks=True
)

print(client.is_ready())  

In [ ]:
qdrant_client = QdrantClient(
    url = qdrant_url,
    api_key = qdrant_api_key
)
print(qdrant_client.get_collections())

In [ ]:
# Config for embedding model

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2" # for sementic memory
)

embedding_model_2 = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5" # for External RAG
)

In [ ]:
# Setting up Vector Store for sementic memory

vectorStore = WeaviateVectorStore(
    client=client,
    index_name="Memory",
    text_key="content",
    embedding=embedding_model
)

In [ ]:
# Setting up Vector Store for External RAG

knowledge_vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name="KnowledgeBase",
    embedding=embedding_model_2,
    content_payload_key="text"
)

In [ ]:
# MAIN CONVERSATIONAL MODEL
llm = ChatGroq(
    model = "llama-3.3-70b-versatile",
    api_key = groq_api_key,
    temperature=0.7
)


# MEMORY RETRIEVER (WEAVIATE)
memory_retriever = vectorStore.as_retriever(
    search_kwargs={
        "k": 3,
    }
)

# KNOWLEDGE RETRIEVER (QDRANT)
knowledge_retriever = knowledge_vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 5,
        "score_threshold": 0.2
    }
)


# COMBINED RETRIEVER
class CombinedRetriever(BaseRetriever):
    user_id: str = Field()
    def _get_relevant_documents(
        self,
        query: str
    ):
        memory_docs = vectorStore.similarity_search(
            query=query,
            k=3,
            filters=Filter.by_property(
                "user_id"
                ).equal(self.user_id)
        )
        knowledge_docs = knowledge_retriever.invoke(query)
        for doc in memory_docs:
            doc.metadata["source_type"] = "memory"
        for doc in knowledge_docs:
            doc.metadata["source_type"] = "knowledge"
        return memory_docs + knowledge_docs


# USER SCOPED RETRIEVER
combined_retriever = CombinedRetriever(
    user_id="rohan_123"
)


# HISTORY AWARE QUERY REWRITING
context_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
Given a chat history and the latest user question,
rewrite the question so it can be understood independently.
"""
    ),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])


history_aware_retriever = create_history_aware_retriever(
    llm,
    combined_retriever,
    context_prompt
)


# MAIN QA PROMPT
stuffed_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a personalized AI engineering assistant.

Use the retrieved context for:
- factual grounding
- personalization

The retrieved context may contain:
1. User semantic memory
2. Technical knowledge chunks

If answer is not present in context,
say you don't know.

Context:
{context}
"""
    ),

    MessagesPlaceholder("chat_history"),

    ("human", "{input}")
])


# DOCUMENT STUFFING CHAIN
answer_chain = create_stuff_documents_chain(
    llm,
    stuffed_prompt
)


# FULL RAG CHAIN
rag_chain = create_retrieval_chain(
    history_aware_retriever,
    answer_chain
)


# REDIS CHAT HISTORY
def get_session_history(session_id: str):
    return RedisChatMessageHistory(
        session_id=session_id,
        url = redis_url
    )


# CONVERSATIONAL RAG PIPELINE
conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer"
)

In [ ]:
# Semantic memory Extractor

memory_llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key = groq_api_key,
    temperature=0
)

class Memory(BaseModel):

    content: str = Field(
        description="Important semantic memory extracted from conversation"
    )

    memory_type: Literal[
        "preference",
        "skill",
        "interest",
        "goal",
        "project"
    ] = Field(
        description="Type of memory"
    )

class MemoryExtraction(BaseModel):

    should_store: bool = Field(
        description="Whether conversation contains important memory"
    )

    memories: List[Memory] = Field(
        description="List of extracted semantic memories"
    )

structured_memory_llm = memory_llm.with_structured_output(
    MemoryExtraction
)

memory_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a semantic memory extraction system.

Extract ONLY important long-term user information.

Store:
- preferences
- skills
- interests
- goals
- ongoing projects

Ignore:
- greetings
- temporary discussion
- casual conversation

IMPORTANT RULES:
- Memories must be fully self-contained sentences.
- Memories must always start with "User".
- Memories must be semantically meaningful and retrieval-friendly.
- Do not store vague keywords.
- Rewrite extracted memories into natural semantic statements.

Good examples:
- "User has interest in deep learning"
- "User prefers C++ for coding"
- "User is learning distributed AI systems"

Bad examples:
- "deep learning"
- "C++"
- "distributed systems"
"""
    ),

    ("human", "{input}")
])

memory_chain = memory_prompt | structured_memory_llm

In [ ]:
# For Testing Purpose

# response = memory_chain.invoke({
#    "input": "I love deep learning and prefer coding in C++."
#})
# print(response.model_dump_json())

In [ ]:
# Checks whether related sementic memory exist in vector DB 

def memory_exists(memory_text, user_id, threshold=0.90):
    results = vectorStore.similarity_search_with_score(
        query=memory_text,
        k=1,
        filters=Filter.by_property("user_id").equal(user_id)
    )
    if not results:
        return False
    document, score = results[0]

    # print("MATCH:", document.page_content)
    # print("SCORE:", score)

    return score < threshold

In [ ]:
#Storing semantic user memory to Vector DB

def store_memories(memories, user_id):
    texts = []
    metadatas = []
    for memory in memories:
        if memory_exists(
            memory.content,
            user_id
        ):

            # print(f"Skipping duplicate: {memory.content}")

            continue
        texts.append(memory.content)
        metadatas.append({
            "memory_type": memory.memory_type,
            "user_id": user_id
        })
    if texts:

        vectorStore.add_texts(
            texts=texts,
            metadatas=metadatas
        )

In [ ]:
# Testing the Pipeline

def Test(prompt, user_id, session_id):
    Answer = conversational_rag_chain.invoke(
        {"input": prompt},
        config={
            "configurable": {
                "session_id": session_id
            }
        }
    )
    display(Markdown(Answer["answer"]))
    response = memory_chain.invoke({
        "input": prompt
    })

    if response.should_store:
        store_memories(
            response.memories,
            user_id=user_id
        )


In [ ]:
Test(prompt="Can you suggest me a project",user_id="rohan_123",session_id="session_1")